<a href="https://colab.research.google.com/github/netsetos/genai-engg-gcp-learners/blob/main/module-09-multimodal-and-pretrained/lesson-9.3-pretrained-apis/practice/GCP_Capstone_9.3_Practice_Lab.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Practice Lab 9.3 — Pre-trained APIs

Runnable companion to the published practice lab: every exercise with a complete solution grounded in the lesson notebook. Run the **Setup** cell first, then work through the exercises. Cloud Shell / `gcloud` steps are `%%bash` cells.

---

## Setup: auth, clients, and shared config

Run this **first**. It installs the pre-trained API SDKs, authenticates with Application Default Credentials (no API keys), and initializes every client the exercises below reuse. Set `PROJECT` to your own project id.

In [ ]:
!pip install -q google-cloud-vision google-cloud-language google-cloud-translate google-cloud-documentai google-genai Pillow

from google.cloud import vision, language_v1, translate_v3 as translate
from google.api_core import retry, exceptions
from google.colab import auth
from google import genai
from google.genai import types
import urllib.request

auth.authenticate_user()  # ADC on Colab — never API keys

PROJECT = 'documind-ai-YOUR-ID'   # <-- replace with your project id
LOCATION = 'us-central1'          # course region (asia-south1 for India prod)
USD_INR = 85                      # for INR cost displays

# Clients reused across every exercise
vision_client = vision.ImageAnnotatorClient()
language_client = language_v1.LanguageServiceClient()
translate_client = translate.TranslationServiceClient()
gen_client = genai.Client(enterprise=True, project=PROJECT, location=LOCATION)

print('Pre-trained API clients ready')

## Exercise 1: Vision OCR on an invoice

**Difficulty:** Easy

Use `DOCUMENT_TEXT_DETECTION` on a scanned invoice image. Extract `full_text` and print block confidence.

1. Download a sample receipt/invoice image.
2. Call `document_text_detection` (dense-document OCR).
3. Print the extracted full text.
4. Walk the Page → Block hierarchy and print block-level confidence.

**Expected:** Full text extracted, block-level confidence scores printed.

In [ ]:
# DOCUMENT_TEXT_DETECTION for dense document OCR
url = 'https://upload.wikimedia.org/wikipedia/commons/0/0b/ReceiptSwiss.jpg'
urllib.request.urlretrieve(url, 'receipt.jpg')

with open('receipt.jpg', 'rb') as f:
    image = vision.Image(content=f.read())

response = vision_client.document_text_detection(image=image)
print('=== Full Text ===')
print(response.full_text_annotation.text[:500])

# Structured hierarchy: Page -> Block -> Paragraph -> Word -> Symbol
print('\n=== Block Confidence ===')
for page in response.full_text_annotation.pages:
    for i, block in enumerate(page.blocks[:3]):
        print(f'Block {i}: confidence={block.confidence:.2%}')

# SafeSearch content moderation (free when bundled)
safe = vision_client.safe_search_detection(image=image).safe_search_annotation
print('\n=== SafeSearch ===')
print(f'Adult: {safe.adult}, Violence: {safe.violence}')

## Exercise 2: Entity extraction

**Difficulty:** Easy

Use the NL API `analyze_entities` on an English contract paragraph. List entities with salience scores.

1. Build a `language_v1.Document` from the text (PLAIN_TEXT).
2. Call `analyze_entities` with UTF8 encoding.
3. Print each entity name, type, and salience.
4. (Bonus) Run `analyze_sentiment` on the same document.

**Expected:** Entities with types (PERSON, ORG, LOCATION) and salience 0-1.

> Note: the NL API supports English (and a few others) only — Indian languages are not supported. Exercise 7 shows the workaround.

In [ ]:
text = '''Sundar Pichai announced at Google I/O in Mountain View that
Alphabet will invest $5 billion in AI infrastructure during Q3 2024.
The CEO emphasized collaboration with TSMC and Samsung.'''

document = language_v1.Document(
    content=text,
    type_=language_v1.Document.Type.PLAIN_TEXT,
)

response = language_client.analyze_entities(
    document=document,
    encoding_type=language_v1.EncodingType.UTF8,
)

print('=== Entities ===')
for entity in response.entities:
    type_name = language_v1.Entity.Type(entity.type_).name
    print(f'{entity.name:30} [{type_name:15}] salience={entity.salience:.3f}')

# Sentiment on the same document
s = language_client.analyze_sentiment(document=document).document_sentiment
print(f'\n=== Sentiment ===')
print(f'Score: {s.score:.3f}, Magnitude: {s.magnitude:.3f}')

## Exercise 3: Translate to Hindi

**Difficulty:** Easy

Use the Translation API to translate English to Hindi. Then detect the language of a Hindi string.

1. Build the `global`-location parent resource path.
2. Call `translate_text` with `target_language_code='hi'`.
3. Print the Hindi translation.
4. Call `detect_language` on a Hindi string and print code + confidence.

**Expected:** Hindi translation output, detected `hi` with confidence.

In [ ]:
parent = f'projects/{PROJECT}/locations/global'
text_en = 'The quarterly earnings exceeded expectations by 15%.'

# English -> Hindi
response = translate_client.translate_text(
    contents=[text_en],
    target_language_code='hi',
    source_language_code='en',
    parent=parent,
)
print('EN:', text_en)
print('HI:', response.translations[0].translated_text)

# Detect language of a Hindi string
detected = translate_client.detect_language(
    parent=parent,
    content='नमस्कार, आप कैसे हैं?',
)
print('\n=== Detection ===')
for lang in detected.languages:
    print(f'Detected: {lang.language_code} ({lang.confidence:.0%})')

## Exercise 4: Combined NL analysis

**Difficulty:** Medium

Use `annotateText` to extract entities + sentiment + classification in one call.

1. Build an `AnnotateTextRequest.Features` with all three flags on.
2. Call `annotate_text` with the document + features + UTF8 encoding.
3. Print entity count, sentiment score, and any classification categories.
4. Handle the case where `classify_text` needs 20+ tokens.

**Expected:** Single response with all three analyses populated.

> Reuses `document` and `language_client` from Exercise 2.

In [ ]:
features = language_v1.AnnotateTextRequest.Features(
    extract_entities=True,
    extract_document_sentiment=True,
    classify_text=True,
)

try:
    response = language_client.annotate_text(
        request={'document': document, 'features': features,
                 'encoding_type': language_v1.EncodingType.UTF8}
    )

    print('=== Entities ===', len(response.entities))
    print('=== Sentiment ===', response.document_sentiment.score)
    if response.categories:
        print('=== Categories ===')
        for cat in response.categories:
            print(f'  {cat.name}: {cat.confidence:.2%}')
except Exception as e:
    print(f'Note: classify_text needs 20+ tokens. Error: {e}')

## Exercise 5: Document AI Invoice Parser

**Difficulty:** Medium

Configure the Invoice Parser. Extract `invoice_id`, totals, and line items from a PDF.

1. Create an **Invoice Parser** processor in the console (Document AI → Processors) and copy its id.
2. Build a `DocumentProcessorServiceClient` for the processor's region.
3. Send the PDF bytes as a `RawDocument` (`application/pdf`).
4. Read the returned `document.entities` — the parser labels `invoice_id`, `total_amount`, `line_item/*`, etc.

**Expected:** Structured entities: `invoice_id`, `total_amount`, `line_items`.

> The lesson notebook does not ship a Document AI cell — this is the standard `documentai_v1` process flow. Create the processor once, then set `PROCESSOR_ID` and `DOCAI_LOCATION` below.

In [ ]:
from google.cloud import documentai_v1 as documentai

# One-time setup: create an 'Invoice Parser' processor in the console,
# then paste its id here. Document AI runs in 'us' or 'eu' (not us-central1).
PROCESSOR_ID = 'YOUR_INVOICE_PROCESSOR_ID'
DOCAI_LOCATION = 'us'

docai_client = documentai.DocumentProcessorServiceClient(
    client_options={'api_endpoint': f'{DOCAI_LOCATION}-documentai.googleapis.com'}
)
processor_name = docai_client.processor_path(PROJECT, DOCAI_LOCATION, PROCESSOR_ID)

def parse_invoice(pdf_path):
    with open(pdf_path, 'rb') as f:
        raw = documentai.RawDocument(content=f.read(), mime_type='application/pdf')
    request = documentai.ProcessRequest(name=processor_name, raw_document=raw)
    result = docai_client.process_document(request=request)
    doc = result.document

    fields, line_items = {}, []
    for ent in doc.entities:
        if ent.type_ == 'line_item':
            line_items.append(ent.mention_text)
        else:
            fields[ent.type_] = ent.mention_text
    return {
        'invoice_id': fields.get('invoice_id'),
        'total_amount': fields.get('total_amount'),
        'line_items': line_items,
        'all_fields': fields,
    }

# Uncomment once you have a processor id and a sample invoice PDF:
# print(parse_invoice('sample_invoice.pdf'))
print('Invoice parser configured. Set PROCESSOR_ID, then call parse_invoice(pdf_path).')

## Exercise 6: Translation LLM vs NMT

**Difficulty:** Medium

Translate the same text with the default NMT model and the **Translation LLM**. Compare quality for technical content.

1. Translate with the default NMT model (no `model` argument).
2. Translate again with `model=.../models/general/translation-llm`.
3. Print both outputs side-by-side.
4. Judge which reads more naturally for the technical sentence.

**Expected:** Both translations compared side-by-side.

> The Translation LLM is invoked by passing a `model` resource path to `translate_text`. It is available in `us-central1` (not `global`), so this call uses a region-scoped parent.

In [ ]:
tech_text = ('Roll back the canary deployment if the p99 latency regresses '
             'beyond the error budget after the blue-green cutover.')
target = 'hi'

# NMT (default model) — runs in the global location
nmt_parent = f'projects/{PROJECT}/locations/global'
nmt = translate_client.translate_text(
    contents=[tech_text], source_language_code='en',
    target_language_code=target, parent=nmt_parent,
)

# Translation LLM — region-scoped parent + model path
llm_location = 'us-central1'
llm_parent = f'projects/{PROJECT}/locations/{llm_location}'
llm_model = f'projects/{PROJECT}/locations/{llm_location}/models/general/translation-llm'
llm = translate_client.translate_text(
    contents=[tech_text], source_language_code='en',
    target_language_code=target, parent=llm_parent, model=llm_model,
)

print('SOURCE :', tech_text)
print('\nNMT    :', nmt.translations[0].translated_text)
print('LLM    :', llm.translations[0].translated_text)
print('\nFor technical/idiomatic content the Translation LLM usually reads more fluently.')

## Exercise 7: Telugu invoice pipeline

**Difficulty:** Challenge

Vision OCR Telugu text → Translation to English → NL API entities. Or Vision → Gemini direct.

1. **v1 (layered):** Vision OCR (supports 10 Indian languages) → Translate to English → NL API `analyze_entities`.
2. **v2 (Gemini direct):** Vision OCR → Gemini extracts entities natively (100+ languages, handles code-mixed text).
3. Run both and compare call count and output.

**Expected:** Two working pipelines. v2 (Gemini direct) is simpler.

In [ ]:
def indian_language_pipeline_v1(image_bytes):
    '''Layered: Vision OCR -> Translate -> NL API entities.'''
    # Step 1: Vision AI OCR (10 Indian languages supported)
    img = vision.Image(content=image_bytes)
    source_text = vision_client.document_text_detection(image=img).full_text_annotation.text

    # Step 2: Translate to English
    parent = f'projects/{PROJECT}/locations/global'
    english_text = translate_client.translate_text(
        contents=[source_text], target_language_code='en', parent=parent,
    ).translations[0].translated_text

    # Step 3: NL API entities (English only)
    doc = language_v1.Document(
        content=english_text, type_=language_v1.Document.Type.PLAIN_TEXT)
    entities = language_client.analyze_entities(document=doc)
    return {
        'source_text': source_text[:200],
        'english_text': english_text[:200],
        'entities': [(e.name, e.salience) for e in entities.entities[:5]],
    }

def indian_language_pipeline_v2(image_bytes):
    '''Simpler: Vision OCR -> Gemini direct (handles 100+ langs).'''
    img = vision.Image(content=image_bytes)
    source_text = vision_client.document_text_detection(image=img).full_text_annotation.text
    response = gen_client.models.generate_content(
        model='gemini-3.6-flash',
        contents=f'Extract entities (person, organization, location, date, amount) '
                 f'from this text as JSON:\n\n{source_text}',
        config=types.GenerateContentConfig(
            response_mime_type='application/json', temperature=0.1),
    )
    return {'source_text': source_text[:200], 'entities_json': response.text}

print('Two approaches defined:')
print('  v1: Vision -> Translation -> NL API (3 API calls)')
print('  v2: Vision -> Gemini direct (2 API calls, handles code-mixed)')

# Try v2 on the receipt from Exercise 1:
with open('receipt.jpg', 'rb') as f:
    print('\n', indian_language_pipeline_v2(f.read())['entities_json'][:300])

## Exercise 8: Full DocuMind pipeline

**Difficulty:** Challenge

A classifier routes each document: invoice → Document AI, English doc → Vision + NL + Gemini, Indian-language doc → OCR + Gemini. Each API call is wrapped with retry protection.

1. Define a shared `RETRY_CONFIG` (retry 429/500/503/timeout; never retry other 4xx).
2. Classify the incoming document (invoice / english / indian).
3. Route to the matching handler and reuse the pipelines from earlier exercises.
4. Return a single structured result.

**Expected:** End-to-end routing with retry protection on each API call.

> The retry config is lifted from lesson Cell 6; the router that ties the exercises together is written fresh for this lab.

In [ ]:
# Retry config from lesson Cell 6 — shared across every pre-trained API call
RETRY_CONFIG = retry.Retry(
    initial=1.0, maximum=60.0, multiplier=2.0, deadline=300.0,
    predicate=retry.if_exception_type(
        exceptions.ServiceUnavailable,    # 503
        exceptions.DeadlineExceeded,      # 408
        exceptions.ResourceExhausted,     # 429 rate limit
        exceptions.InternalServerError,   # 500
    ),
)

def safe_ocr(image_bytes):
    img = vision.Image(content=image_bytes)
    try:
        return vision_client.document_text_detection(
            image=img, retry=RETRY_CONFIG).full_text_annotation.text
    except exceptions.InvalidArgument as e:   # never retry a bad 4xx input
        print(f'Invalid input: {e}')
        return ''

def documind_route(image_bytes, doc_kind):
    '''doc_kind: "invoice" | "english" | "indian".'''
    if doc_kind == 'invoice':
        # Structured extraction via Document AI (see Exercise 5)
        return {'route': 'document_ai',
                'note': 'call parse_invoice(pdf_path) with a configured processor'}

    if doc_kind == 'english':
        text = safe_ocr(image_bytes)
        doc = language_v1.Document(
            content=text, type_=language_v1.Document.Type.PLAIN_TEXT)
        ents = language_client.analyze_entities(document=doc, retry=RETRY_CONFIG)
        return {'route': 'vision+nl',
                'entities': [(e.name, e.salience) for e in ents.entities[:5]]}

    if doc_kind == 'indian':
        # OCR + Gemini direct (see Exercise 7 v2)
        return {'route': 'vision+gemini', **indian_language_pipeline_v2(image_bytes)}

    raise ValueError(f'unknown doc_kind: {doc_kind}')

# Demo: route the Exercise 1 receipt as an English doc
with open('receipt.jpg', 'rb') as f:
    result = documind_route(f.read(), 'english')
print(result)